In [3]:
import torch
import torchvision
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())
import sys
!{sys.executable} -m pip install opencv-python matplotlib
!{sys.executable} -m pip install 'git+https://github.com/facebookresearch/segment-anything.git'

!mkdir images
!wget -P images https://raw.githubusercontent.com/facebookresearch/segment-anything/main/notebooks/images/dog.jpg
    
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

PyTorch version: 2.1.0+cu118
Torchvision version: 0.16.0+cu118
CUDA is available: True
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 4.5 MB/s eta 0:00:0000:0100:020m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 30.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.5 MB/s eta 0:00:00
  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-25xgcurx
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-25xgcurx
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
  Created wheel for segment-anything: filename=segment_anything-1.0-py3-none-any.whl size=36588 sha256=75f3c201ac852bc564298aca5a7d9617974625a73162da50048b4c55b24e2198
  Stored in direc

In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import cv2

In [11]:
def show_anns(anns):
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)

    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    img[:,:,3] = 0
    
    # Add numbers to each mask
    for idx, ann in enumerate(sorted_anns):
        m = ann['segmentation']
        color_mask = np.concatenate([np.random.random(3), [0.35]])
        img[m] = color_mask
        
        # Calculate center of mass for the mask
        y_indices, x_indices = np.where(m)
        if len(y_indices) > 0 and len(x_indices) > 0:
            center_y = int(np.mean(y_indices))
            center_x = int(np.mean(x_indices))
            
            # Add text with white color and black edge
            plt.text(center_x, center_y, str(idx+1), 
                    fontsize=12, 
                    color='white',
                    bbox=dict(facecolor='black', alpha=0.5),
                    ha='center', 
                    va='center')
    
    ax.imshow(img)

In [12]:
image = cv2.imread('me.png')
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

import sys
sys.path.append("..")
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"

device = "cuda"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

mask_generator = SamAutomaticMaskGenerator(sam)



masks = mask_generator.generate(image)

plt.figure(figsize=(20,20))
plt.imshow(image)
show_anns(masks)
plt.axis('off')
# plt.show()  # Remove or comment out the show command
plt.savefig('output_image.png', bbox_inches='tight', pad_inches=0)
plt.close() 

In [13]:
def show_selected_masks(image, masks, selected_numbers):
    """
    Display image with only selected mask numbers
    Args:
        image: Original image
        masks: List of mask dictionaries from SAM
        selected_numbers: List of numbers to display (1-based indexing as shown in the numbered image)
    """
    # Convert to 0-based indexing
    selected_indices = [n-1 for n in selected_numbers]
    
    # Sort masks by area as in original show_anns
    sorted_anns = sorted(masks, key=(lambda x: x['area']), reverse=True)
    
    # Filter only selected masks
    filtered_masks = [mask for idx, mask in enumerate(sorted_anns) if idx in selected_indices]
    
    plt.figure(figsize=(20,20))
    plt.imshow(image)
    
    if len(filtered_masks) == 0:
        return
    
    ax = plt.gca()
    ax.set_autoscale_on(False)

    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    img[:,:,3] = 0
    
    # Show only selected masks with their original numbers
    for idx, ann in enumerate(filtered_masks):
        m = ann['segmentation']
        color_mask = np.concatenate([np.random.random(3), [0.35]])
        img[m] = color_mask
        
        # Calculate center of mass for the mask
        y_indices, x_indices = np.where(m)
        if len(y_indices) > 0 and len(x_indices) > 0:
            center_y = int(np.mean(y_indices))
            center_x = int(np.mean(x_indices))
            
            # Show original number (selected_indices[idx] + 1)
            plt.text(center_x, center_y, str(selected_numbers[idx]), 
                    fontsize=12, 
                    color='white',
                    bbox=dict(facecolor='black', alpha=0.5),
                    ha='center', 
                    va='center')
    
    ax.imshow(img)
    plt.axis('off')
    plt.savefig('filtered_masks.png', bbox_inches='tight', pad_inches=0)
    plt.close()


In [14]:
# Then generate image with selected masks only
# Example: show only masks 1, 3, and 5
selected_numbers = [4, 15]  # Change these numbers as needed
show_selected_masks(image, masks, selected_numbers)

In [17]:
def create_binary_mask(image, masks, selected_numbers):
    """
    Create a binary mask where selected masks are white (255) and everything else is black (0)
    Args:
        image: Original image
        masks: List of mask dictionaries from SAM
        selected_numbers: List of numbers to make white (1-based indexing)
    """
    # Convert to 0-based indexing
    selected_indices = [n-1 for n in selected_numbers]
    
    # Sort masks by area as in original show_anns
    sorted_anns = sorted(masks, key=(lambda x: x['area']), reverse=True)
    
    # Create black background
    binary_mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
    
    # Make selected masks white
    for idx in selected_indices:
        if idx < len(sorted_anns):
            m = sorted_anns[idx]['segmentation']
            binary_mask[m] = 255
    
    # Save the binary mask
    plt.figure(figsize=(20,20))
    plt.imshow(binary_mask, cmap='gray')
    plt.axis('off')
    plt.savefig('binary_mask.png', bbox_inches='tight', pad_inches=0)
    plt.close()
    
    return binary_mask

In [18]:
# Add this after your existing code:
selected_numbers = [4, 15]  # Change these numbers as needed
binary_mask = create_binary_mask(image, masks, selected_numbers)